# MODEL COMPARISON & DASHBOARD DATA EXPORT

**Notebook 06**: Compare all models and prepare data for dashboard

This notebook:
1. Loads results from ARIMA, Prophet, LSTM
2. Compares performance
3. Selects best model per series
4. Exports data for Antigravity dashboard

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Libraries imported")

Libraries imported


## Load All Model Results

In [2]:
# Load results from each model
arima_results = pd.read_csv('../reports/model_comparison/arima_results.csv')
prophet_results = pd.read_csv('../reports/model_comparison/prophet_results.csv')

try:
    lstm_results = pd.read_csv('../reports/model_comparison/lstm_results.csv')
except:
    lstm_results = pd.DataFrame()
    print("Note: LSTM results not found, continuing with ARIMA and Prophet only")

print(f"ARIMA models: {len(arima_results)}")
print(f"Prophet models: {len(prophet_results)}")
print(f"LSTM models: {len(lstm_results) if len(lstm_results) > 0 else 0}")

ARIMA models: 60
Prophet models: 60
LSTM models: 30


## Overall Performance Comparison

In [3]:
# Calculate average metrics
comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'LSTM'],
    'Avg MAPE': [
        arima_results['mape'].mean(),
        prophet_results['mape'].mean(),
        lstm_results['mape'].mean() if len(lstm_results) > 0 else np.nan
    ],
    'Avg MAE': [
        arima_results['mae'].mean(),
        prophet_results['mae'].mean(),
        lstm_results['mae'].mean() if len(lstm_results) > 0 else np.nan
    ],
    'Avg RMSE': [
        arima_results['rmse'].mean(),
        prophet_results['rmse'].mean(),
        lstm_results['rmse'].mean() if len(lstm_results) > 0 else np.nan
    ],
    'N Models': [
        len(arima_results),
        len(prophet_results),
        len(lstm_results) if len(lstm_results) > 0 else 0
    ]
})

print("\nMODEL COMPARISON SUMMARY")
print("=" * 70)
print(comparison.to_string(index=False))

# Identify best model
comparison_clean = comparison.dropna()
best_model = comparison_clean.loc[comparison_clean['Avg MAPE'].idxmin(), 'Model']
print(f"\nBest Overall Model: {best_model}")


MODEL COMPARISON SUMMARY
  Model  Avg MAPE   Avg MAE  Avg RMSE  N Models
  ARIMA  7.876439 14.912083 18.718499        60
Prophet 15.071204 19.078729 21.635116        60
   LSTM 13.647725 46.071491 46.071491        30

Best Overall Model: ARIMA


## Visualization: Model Performance

In [4]:
# Create comparison bar chart
fig = go.Figure()

models = comparison_clean['Model'].tolist()
mapes = comparison_clean['Avg MAPE'].tolist()

fig.add_trace(go.Bar(
    x=models,
    y=mapes,
    marker=dict(color=['#FF6B6B', '#4ECDC4', '#45B7D1']),
    text=[f"{m:.2f}%" for m in mapes],
    textposition='auto'
))

fig.update_layout(
    title='Model Performance Comparison (Lower is Better)',
    xaxis_title='Model',
    yaxis_title='Average MAPE (%)',
    template='plotly_white',
    height=500
)

# fig.show()
fig.write_html('../reports/model_comparison/performance_comparison.html')
print("Saved: reports/model_comparison/performance_comparison.html")

Saved: reports/model_comparison/performance_comparison.html


## Select Best Model Per Series

In [5]:
# Combine all results
arima_results['model_type'] = 'ARIMA'
prophet_results['model_type'] = 'Prophet'

if len(lstm_results) > 0:
    lstm_results['model_type'] = 'LSTM'
    all_results = pd.concat([arima_results, prophet_results, lstm_results], ignore_index=True)
else:
    all_results = pd.concat([arima_results, prophet_results], ignore_index=True)

# For each series, find best model
best_models = all_results.loc[
    all_results.groupby(['state', 'sector', 'fuel'])['mape'].idxmin()
].reset_index(drop=True)

print(f"\nSelected best model for {len(best_models)} series")
print("\nModel selection distribution:")
print(best_models['model_type'].value_counts())


Selected best model for 60 series

Model selection distribution:
model_type
ARIMA      47
Prophet     9
LSTM        4
Name: count, dtype: int64


## Prepare Dashboard Data

Export everything Antigravity needs

In [6]:
# Load historical data
historical = pd.read_csv('../data/processed/emissions_processed.csv')

# Load predictions
arima_pred = pd.read_csv('../data/dashboard/arima_predictions_2022_2030.csv')
prophet_pred = pd.read_csv('../data/dashboard/prophet_predictions_2022_2030.csv')

print("Loaded historical and prediction data")

Loaded historical and prediction data


In [7]:
# Create dashboard-ready JSON files

# 1. Historical data summary (by state, year)
historical_summary = historical[
    historical['fuel-name'] == 'All Fuels'
].groupby(['year', 'state-name'])['value'].sum().reset_index()

historical_summary.columns = ['year', 'state', 'emissions']
historical_summary.to_json('../data/dashboard/historical_emissions.json', orient='records')
print("Exported: historical_emissions.json")

# 2. Model performance summary
model_summary = {
    'overall_comparison': comparison.to_dict('records'),
    'best_model': best_model,
    'total_models_trained': len(all_results),
    'arima': {
        'count': len(arima_results),
        'avg_mape': float(arima_results['mape'].mean()),
        'best_mape': float(arima_results['mape'].min())
    },
    'prophet': {
        'count': len(prophet_results),
        'avg_mape': float(prophet_results['mape'].mean()),
        'best_mape': float(prophet_results['mape'].min())
    }
}

if len(lstm_results) > 0:
    model_summary['lstm'] = {
        'count': len(lstm_results),
        'avg_mape': float(lstm_results['mape'].mean()),
        'best_mape': float(lstm_results['mape'].min())
    }

with open('../data/dashboard/model_summary.json', 'w') as f:
    json.dump(model_summary, f, indent=4)
print("Exported: model_summary.json")

# 3. Combined predictions (best model for each series)
all_predictions = pd.concat([arima_pred, prophet_pred], ignore_index=True)
all_predictions.to_json('../data/dashboard/future_predictions.json', orient='records')
print("Exported: future_predictions.json")

# 4. Top states info
top_states_2021 = historical[
    (historical['year'] == 2021) & 
    (historical['fuel-name'] == 'All Fuels')
].groupby('state-name')['value'].sum().nlargest(10)

top_states_info = {
    'top_10_states': [
        {'state': state, 'emissions': float(value)}
        for state, value in top_states_2021.items()
    ]
}

with open('../data/dashboard/top_states.json', 'w') as f:
    json.dump(top_states_info, f, indent=4)
print("Exported: top_states.json")

Exported: historical_emissions.json
Exported: model_summary.json
Exported: future_predictions.json
Exported: top_states.json


## Create Dashboard Configuration File

In [8]:
dashboard_config = {
    'project_name': 'Carbon Emission Prediction Platform',
    'data_files': {
        'historical': 'data/dashboard/historical_emissions.json',
        'predictions': 'data/dashboard/future_predictions.json',
        'model_summary': 'data/dashboard/model_summary.json',
        'top_states': 'data/dashboard/top_states.json'
    },
    'time_range': {
        'historical_start': 1970,
        'historical_end': 2021,
        'forecast_start': 2022,
        'forecast_end': 2030
    },
    'models': {
        'available': ['ARIMA', 'Prophet', 'LSTM'] if len(lstm_results) > 0 else ['ARIMA', 'Prophet'],
        'default': best_model
    },
    'filters': {
        'states': historical['state-name'].unique().tolist(),
        'sectors': historical['sector-name'].unique().tolist(),
        'fuels': historical['fuel-name'].unique().tolist()
    }
}

with open('../data/dashboard/dashboard_config.json', 'w') as f:
    json.dump(dashboard_config, f, indent=4)

print("\nExported: dashboard_config.json")
print("\nThis file tells Antigravity everything about your data!")


Exported: dashboard_config.json

This file tells Antigravity everything about your data!


## Summary Report

In [9]:
summary_report = f"""
{'='*70}
MODEL TRAINING & COMPARISON COMPLETE
{'='*70}

MODELS TRAINED:
  - ARIMA: {len(arima_results)} models (Avg MAPE: {arima_results['mape'].mean():.2f}%)
  - Prophet: {len(prophet_results)} models (Avg MAPE: {prophet_results['mape'].mean():.2f}%)
{'  - LSTM: ' + str(len(lstm_results)) + ' models (Avg MAPE: ' + f"{lstm_results['mape'].mean():.2f}%" + ')' if len(lstm_results) > 0 else ''}

BEST OVERALL MODEL: {best_model}

FILES EXPORTED FOR DASHBOARD:
  data/dashboard/
  ├── historical_emissions.json
  ├── future_predictions.json (2022-2030)
  ├── model_summary.json
  ├── top_states.json
  └── dashboard_config.json

NEXT STEP:
  Give these files to Antigravity to build the dashboard!
  
  Dashboard should have:
  - Interactive filters (state, sector, fuel, year range)
  - Time series chart (historical + forecast)
  - Model performance comparison
  - Top states ranking
  - Export functionality

{'='*70}
"""

print(summary_report)

# Save report
with open('../reports/FINAL_SUMMARY.txt', 'w') as f:
    f.write(summary_report)

print("Saved: reports/FINAL_SUMMARY.txt")


MODEL TRAINING & COMPARISON COMPLETE

MODELS TRAINED:
  - ARIMA: 60 models (Avg MAPE: 7.88%)
  - Prophet: 60 models (Avg MAPE: 15.07%)
  - LSTM: 30 models (Avg MAPE: 13.65%)

BEST OVERALL MODEL: ARIMA

FILES EXPORTED FOR DASHBOARD:
  data/dashboard/
  ├── historical_emissions.json
  ├── future_predictions.json (2022-2030)
  ├── model_summary.json
  ├── top_states.json
  └── dashboard_config.json

NEXT STEP:
  Give these files to Antigravity to build the dashboard!
  
  Dashboard should have:
  - Interactive filters (state, sector, fuel, year range)
  - Time series chart (historical + forecast)
  - Model performance comparison
  - Top states ranking
  - Export functionality


Saved: reports/FINAL_SUMMARY.txt


## Project Complete

All notebooks executed successfully!

Next: Build dashboard with Antigravity using the exported JSON files